# Nucleotide Transformer smoke test

Purpose: check that InstaDeep's repo installs and runs on this Colab runtime (GPU/TPU), and discover the actual output structure of a forward pass — the docs confirm an `embeddings_N` key but do NOT confirm a `logits` key, so this cell finds out empirically rather than assuming.

Set Runtime > Change runtime type > GPU (or TPU) before running.

In [ ]:
!pip install git+https://github.com/instadeepai/nucleotide-transformer.git
!pip install -U "jax[cuda12]" "numpy<2.0" --force-reinstall

**Restart now: Runtime > Restart session.** This cell force-reinstalls jax and numpy — the kernel still has the old binary versions loaded in memory, so nothing after this point is safe to run until you restart. Packages persist on disk across a restart, so do not re-run the cell above afterward — just continue to the next cell.

In [ ]:
import jax
print(jax.devices())

## Load smallest documented-working checkpoint, run one forward pass

Using `"250M_multi_species_v2"` specifically because it's the exact model name shown working in the repo's own docs — not guessing at a smaller variant's name without confirmation. `embeddings_layers_to_save=()` since we don't need embeddings for this check.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import time
import haiku as hk
import jax
import jax.numpy as jnp
from nucleotide_transformer.pretrained import get_pretrained_model

t0 = time.time()
parameters, forward_fn, tokenizer, config = get_pretrained_model(
    model_name="250M_multi_species_v2",
    embeddings_layers_to_save=(),
    max_positions=32,
)
print(f"load time: {time.time() - t0:.1f}s")

sequences = ["ATTCCGATTCCGATTCCG", "ATTTCTCTCTCTCTCTGAGATCGATCGATCGAT"]
tokens_ids = [b[1] for b in tokenizer.batch_tokenize(sequences)]
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

random_key = jax.random.PRNGKey(0)
forward_fn = hk.transform(forward_fn)

t0 = time.time()
outs = forward_fn.apply(parameters, random_key, tokens)
print(f"forward pass time: {time.time() - t0:.1f}s")

print(outs.keys())
for k, v in outs.items():
    print(k, v.shape)